# Steam semantic outputs regeneration

Goal: rebuild semantic artifacts used in the language-feature experiments.

Main steps:
- compute embeddings for all games;
- build semantic features for train/validation/test splits;
- save `item_embeddings.npy`, `*_semantic_features.parquet`, and `*_tabular_semantic.parquet`;
- compute quick semantic-only, tabular, and tabular-plus-semantic metrics.

Expected input files for the semantic bundle:
- `item_metadata.parquet`
- `user_histories_mvp_v4_temporal.parquet`
- `train_tabular.parquet`
- `val_tabular.parquet`
- `test_tabular.parquet`
- `train_item_popularity.csv`, if available


In [ ]:
!pip install -q -U sentence-transformers pandas pyarrow scikit-learn


In [ ]:
import os, glob, json, ast, math, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_recall_fscore_support
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available(): print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 256
QUICK_MODEL_CHECK = True
OUT_DIR = Path("/kaggle/working/steam_semantic_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEM_COLS = [
    "semantic_target_liked_cosine", "semantic_target_disliked_cosine", "semantic_similarity_gap",
    "semantic_max_liked_cosine", "semantic_max_disliked_cosine",
    "semantic_top3_liked_mean_cosine", "semantic_top3_disliked_mean_cosine"
]
print("OUT_DIR:", OUT_DIR)


In [ ]:
def find_file(filename):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not hits: raise FileNotFoundError(f"Не найден {filename} в /kaggle/input")
    return Path(sorted(hits, key=lambda p: len(p))[0])

item_meta_path = find_file("item_metadata.parquet")
histories_path = find_file("user_histories_mvp_v4_temporal.parquet")
train_tab_path = find_file("train_tabular.parquet")
val_tab_path = find_file("val_tabular.parquet")
test_tab_path = find_file("test_tabular.parquet")
print("item_meta:", item_meta_path)
print("histories:", histories_path)
print("train_tab:", train_tab_path)
print("val_tab:", val_tab_path)
print("test_tab:", test_tab_path)

item_meta = pd.read_parquet(item_meta_path)
histories = pd.read_parquet(histories_path)
train_tab = pd.read_parquet(train_tab_path)
val_tab = pd.read_parquet(val_tab_path)
test_tab = pd.read_parquet(test_tab_path)
print("item_meta:", item_meta.shape)
print("histories:", histories.shape)
print("train/val/test:", train_tab.shape, val_tab.shape, test_tab.shape)
print("item columns:", item_meta.columns.tolist()[:60])
print("history columns:", histories.columns.tolist()[:60])
print("tab columns:", train_tab.columns.tolist()[:60])


In [ ]:
def get_col(df, candidates):
    for c in candidates:
        if c in df.columns: return c
    return None

APP_COL = get_col(item_meta, ["app_id", "steam_appid", "id"])
if APP_COL is None: raise ValueError("Не найден app_id column")
TEXT_COL = get_col(item_meta, ["item_text_retrieval", "item_text", "text"])

def safe_str(x, max_chars=None):
    if pd.isna(x): return ""
    if isinstance(x, (list, tuple, set, np.ndarray)): s = ", ".join(map(str, list(x)[:30]))
    else: s = str(x)
    s = " ".join(s.split())
    return s[:max_chars] if max_chars else s

def build_item_text(row):
    if TEXT_COL is not None and safe_str(row.get(TEXT_COL)):
        return safe_str(row.get(TEXT_COL), 1200)
    parts=[]
    title_col = get_col(item_meta, ["title", "name", "game_title"])
    for c in [title_col, "genres", "tags", "categories", "description", "short_description", "about_the_game"]:
        if c and c in row.index:
            val = safe_str(row.get(c), 700 if c in ["description", "short_description", "about_the_game"] else 300)
            if val: parts.append(f"{c}: {val}")
    return " | ".join(parts)[:1500]

item_meta = item_meta.drop_duplicates(APP_COL).copy()
item_meta["app_id_for_embedding"] = item_meta[APP_COL].astype(int)
item_meta["embedding_text"] = item_meta.apply(build_item_text, axis=1)
print("items:", len(item_meta))
print(item_meta[["app_id_for_embedding", "embedding_text"]].head(3).to_string())


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(EMBEDDING_MODEL, device=device)
texts = item_meta["embedding_text"].fillna("").astype(str).tolist()
embeddings = model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
np.save(OUT_DIR / "item_embeddings.npy", embeddings)
item_index = item_meta[["app_id_for_embedding"]].rename(columns={"app_id_for_embedding":"app_id"}).reset_index(drop=True)
item_index["embedding_row"] = np.arange(len(item_index))
item_index.to_parquet(OUT_DIR / "item_embedding_index.parquet", index=False)
item_meta[["app_id_for_embedding","embedding_text"]].rename(columns={"app_id_for_embedding":"app_id"}).to_parquet(OUT_DIR / "item_embedding_texts.parquet", index=False)
print("embeddings:", embeddings.shape)


In [ ]:
app_ids = item_index["app_id"].astype(int).values
app_to_row = {int(app): int(i) for i, app in enumerate(app_ids)}

def parse_list_value(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, (list, tuple)): return list(x)
    if isinstance(x, str):
        s=x.strip()
        if not s: return []
        try:
            val=ast.literal_eval(s)
            return list(val) if isinstance(val, (list, tuple, np.ndarray)) else [val]
        except Exception:
            return [v.strip() for v in s.split(",") if v.strip()]
    return [x]

def to_bool_label(x):
    if isinstance(x, str): return x.strip().lower() in ["1","true","yes","recommended","positive"]
    return bool(x)

def infer_target_col(df):
    for c in ["target_app_id", "target_item_id", "target_game_app_id", "app_id"]:
        if c in df.columns: return c
    raise ValueError("Не найден target/app_id column")

def find_history_source(tab_df, histories):
    if "history_app_ids" in tab_df.columns or "liked_app_ids" in tab_df.columns: return tab_df.copy()
    if "sample_id" in tab_df.columns and "sample_id" in histories.columns:
        cols = [c for c in histories.columns if c not in tab_df.columns or c == "sample_id"]
        return tab_df.merge(histories[cols], on="sample_id", how="left")
    h = histories.iloc[:len(tab_df)].reset_index(drop=True)
    t = tab_df.reset_index(drop=True).copy()
    for c in h.columns:
        if c not in t.columns: t[c]=h[c]
    return t

def get_embedding(app_id):
    try: idx = app_to_row.get(int(app_id))
    except Exception: idx = None
    return None if idx is None else embeddings[idx]

def cos(a,b): return 0.0 if a is None or b is None else float(np.dot(a,b))

def mean_emb(app_list):
    vecs=[get_embedding(x) for x in app_list]
    vecs=[v for v in vecs if v is not None]
    if not vecs: return None
    v=np.mean(vecs,axis=0); n=np.linalg.norm(v)
    return None if n==0 else (v/n).astype("float32")

def topk_mean(vals,k=3): return 0.0 if not vals else float(np.mean(sorted(vals, reverse=True)[:k]))

def extract_liked_disliked(row):
    liked = disliked = None
    for c in ["liked_app_ids","history_liked_app_ids","liked_history_app_ids"]:
        if c in row.index: liked=parse_list_value(row.get(c)); break
    for c in ["disliked_app_ids","history_disliked_app_ids","disliked_history_app_ids"]:
        if c in row.index: disliked=parse_list_value(row.get(c)); break
    if liked is not None or disliked is not None: return liked or [], disliked or []
    app_col = next((c for c in ["history_app_ids","history_apps","history_items"] if c in row.index), None)
    lab_col = next((c for c in ["history_labels","history_is_recommended","history_outputs"] if c in row.index), None)
    if app_col and lab_col:
        apps=parse_list_value(row.get(app_col)); labels=parse_list_value(row.get(lab_col))
        liked=[]; disliked=[]
        for app, lab in zip(apps, labels): (liked if to_bool_label(lab) else disliked).append(app)
        return liked, disliked
    return [], []

def make_semantic_features(tab_df, split_name):
    df=find_history_source(tab_df, histories)
    target_col=infer_target_col(df)
    rows=[]
    for pos, (_, row) in enumerate(df.iterrows(), start=1):
        target_emb=get_embedding(row.get(target_col))
        liked, disliked=extract_liked_disliked(row)
        liked_mean=mean_emb(liked); disliked_mean=mean_emb(disliked)
        liked_cos=cos(target_emb, liked_mean); disliked_cos=cos(target_emb, disliked_mean)
        liked_sims=[cos(target_emb, get_embedding(x)) for x in liked]
        disliked_sims=[cos(target_emb, get_embedding(x)) for x in disliked]
        out={
            "semantic_target_liked_cosine": liked_cos,
            "semantic_target_disliked_cosine": disliked_cos,
            "semantic_similarity_gap": liked_cos-disliked_cos,
            "semantic_max_liked_cosine": float(max(liked_sims)) if liked_sims else 0.0,
            "semantic_max_disliked_cosine": float(max(disliked_sims)) if disliked_sims else 0.0,
            "semantic_top3_liked_mean_cosine": topk_mean(liked_sims,3),
            "semantic_top3_disliked_mean_cosine": topk_mean(disliked_sims,3),
        }
        if "sample_id" in df.columns: out["sample_id"]=row.get("sample_id")
        rows.append(out)
        if pos % 20000 == 0: print(split_name, "processed", pos)
    return pd.DataFrame(rows)

train_sem=make_semantic_features(train_tab,"train")
val_sem=make_semantic_features(val_tab,"val")
test_sem=make_semantic_features(test_tab,"test")
train_sem.to_parquet(OUT_DIR/"train_semantic_features.parquet", index=False)
val_sem.to_parquet(OUT_DIR/"val_semantic_features.parquet", index=False)
test_sem.to_parquet(OUT_DIR/"test_semantic_features.parquet", index=False)
print(train_sem.head())


In [ ]:
def merge_sem(tab, sem):
    if "sample_id" in tab.columns and "sample_id" in sem.columns: return tab.merge(sem,on="sample_id",how="left")
    return pd.concat([tab.reset_index(drop=True), sem.drop(columns=["sample_id"], errors="ignore").reset_index(drop=True)], axis=1)
train_tab_sem=merge_sem(train_tab, train_sem)
val_tab_sem=merge_sem(val_tab, val_sem)
test_tab_sem=merge_sem(test_tab, test_sem)
train_tab_sem.to_parquet(OUT_DIR/"train_tabular_semantic.parquet", index=False)
val_tab_sem.to_parquet(OUT_DIR/"val_tabular_semantic.parquet", index=False)
test_tab_sem.to_parquet(OUT_DIR/"test_tabular_semantic.parquet", index=False)
print(train_tab_sem.shape, val_tab_sem.shape, test_tab_sem.shape)


In [ ]:
def get_label_col(df):
    for c in ["label_strict","is_recommended","label"]:
        if c in df.columns: return c
    raise ValueError("Не найден label column")

def label_to_int(s):
    if s.dtype == bool: return s.astype(int)
    return s.astype(str).str.lower().isin(["1","true","yes","recommended"]).astype(int)

def select_numeric_features(df):
    leak=["label","output","target_hours","review_id","hours_reference","hours_rel","is_low_hours","is_high_hours"]
    cols=[]
    for c in df.columns:
        cl=c.lower()
        if any(p in cl for p in leak): continue
        if pd.api.types.is_numeric_dtype(df[c]): cols.append(c)
    return cols

def best_thr(y,score):
    best=(0.5,-1)
    for thr in np.linspace(0.01,0.99,99):
        pred=(score>=thr).astype(int)
        _,_,f1,_=precision_recall_fscore_support(y,pred,average="binary",zero_division=0)
        if f1>best[1]: best=(float(thr),float(f1))
    return best[0]

def metrics(y,score,thr):
    pred=(score>=thr).astype(int)
    p,r,f1,_=precision_recall_fscore_support(y,pred,average="binary",zero_division=0)
    return {"roc_auc":float(roc_auc_score(y,score)),"pr_auc":float(average_precision_score(y,score)),"accuracy":float(accuracy_score(y,pred)),"precision":float(p),"recall":float(r),"f1":float(f1),"threshold":float(thr)}

if QUICK_MODEL_CHECK:
    label_col=get_label_col(train_tab_sem)
    y_train=label_to_int(train_tab_sem[label_col]); y_val=label_to_int(val_tab_sem[label_col]); y_test=label_to_int(test_tab_sem[label_col])
    variants={
        "semantic_only": SEM_COLS,
        "numeric_safe_without_semantic": [c for c in select_numeric_features(train_tab_sem) if c not in SEM_COLS],
        "numeric_safe_plus_semantic": select_numeric_features(train_tab_sem),
    }
    rows=[]
    for name, cols in variants.items():
        cols=[c for c in cols if c in train_tab_sem.columns]
        print(name, len(cols))
        clf=make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingClassifier(max_iter=150, learning_rate=0.06, l2_regularization=0.05, random_state=SEED))
        clf.fit(train_tab_sem[cols], y_train)
        val_score=clf.predict_proba(val_tab_sem[cols])[:,1]
        test_score=clf.predict_proba(test_tab_sem[cols])[:,1]
        thr=best_thr(y_val, val_score)
        rows.append({"variant":name,"split":"val",**metrics(y_val,val_score,thr)})
        rows.append({"variant":name,"split":"test",**metrics(y_test,test_score,thr)})
    metrics_df=pd.DataFrame(rows)
    metrics_df.to_csv(OUT_DIR/"metrics_semantic_comparison.csv", index=False)
    display(metrics_df)
else:
    metrics_df=pd.DataFrame()


In [ ]:
summary={
    "embedding_model": EMBEDDING_MODEL,
    "num_items": int(len(item_meta)),
    "embedding_dim": int(embeddings.shape[1]),
    "train_rows": int(len(train_tab_sem)),
    "val_rows": int(len(val_tab_sem)),
    "test_rows": int(len(test_tab_sem)),
    "semantic_features": SEM_COLS,
    "quick_model_check": bool(QUICK_MODEL_CHECK),
    "metrics": metrics_df.to_dict(orient="records") if len(metrics_df) else []
}
with open(OUT_DIR/"semantic_run_summary.json","w",encoding="utf-8") as f: json.dump(summary,f,ensure_ascii=False,indent=2)
shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("saved:", OUT_DIR)
print("archive:", str(OUT_DIR)+".zip")
